# Finalize and validate manual relevance labels

Run from top to bottom using a Python 3 kernel. Only the Python standard library is required. This notebook reads a pinned Git snapshot without checking out the P3 branch or modifying submitted annotations. Outputs stay local under `results/`; do not commit human labels, passage text, or executed notebook outputs.

**Scope:** annotation validation, pre-adjudication agreement, human adjudication preparation, and provisional gold-label export. This does not establish that labeling was independent or that the evaluation split was approved. Passage schemas and split mappings remain **v0-proposed** until team confirmation.

**Expected sequence:** validate → measure independent agreement → human adjudication → review provenance and scope → export. Never replace original judgments to increase agreement. Algorithm identities were disclosed in these views; describe them as source-disclosed, not system-blinded.


## 1. Pin inputs

**Expected:** the intended commit, 20 annotation Markdown files, 200 questions and 1,853 unique pairs for this version. A different count must be investigated, not padded. Fetch separately with `git fetch origin p3-qasper-foundation` if needed. Re-executing this cell pins the then-current remote-tracking commit. No network access is performed by this notebook.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import subprocess, json, re, hashlib, csv, math
ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / ".git").exists(), "Start inside the project repository"
def git(*args):
    return subprocess.check_output(["git", "-C", str(ROOT), *args])
COMMIT = git("rev-parse", "origin/p3-qasper-foundation").decode().strip()
BASE = "data/processed/annotation_common_v3"
VIEWS = BASE + "_markdown"
OUT = ROOT / "results" / ("annotation-validation-" + COMMIT[:12])
def read_bytes(path):
    return git("show", f"{COMMIT}:{path}")
def read_text(path):
    return read_bytes(path).decode("utf-8")
def jsonl(path):
    return [json.loads(line) for line in read_text(path).splitlines() if line.strip()]
paths = git("ls-tree", "-r", "--name-only", COMMIT, VIEWS).decode().splitlines()
markdown = sorted(p for p in paths if p.endswith(".md"))
pairs = jsonl(BASE + "/pairs.jsonl")
manifest = json.loads(read_text(BASE + "/manifest.json"))
view_manifest = json.loads(read_text(VIEWS + "/manifest.json"))
assert len(markdown) == 20
assert len(pairs) == 1853
assert len({(p["question_id"], p["passage_id"]) for p in pairs}) == len(pairs)
assert len({p["question_id"] for p in pairs}) == 200
assert len({p["record_number"] for p in pairs}) == len(pairs)
print("Pinned commit:", COMMIT)
print("Questions:", 200, "Unique pairs:", len(pairs), "Files:", len(markdown))
print("Output directory (not yet created):", OUT)


## 2. Validate integrity, record mapping and completeness

**Expected:** 1,853 binary judgments in each slot, no missing/duplicate record IDs, two different named people per pair, and unchanged question, passage, attribution, and record order. Original export output hashes describe blank templates, so human-edited files must not match those hashes. Instead, verify frozen pool input hashes and compare document bodies after masking names, labels, and quoted rationale notes immediately following labels against pristine templates recovered from Git history.

A failure stops the notebook. Investigate the source; do not silently fix or drop a record. Identity spelling aliases require a documented review before merging names. Matching names and distinct slots cannot prove independent work: confirm that procedurally with the annotators.


In [ ]:
# Verify source pool bytes against the export's frozen input hashes.
for path, expected in view_manifest["inputs"].items():
    assert hashlib.sha256(read_bytes(path)).hexdigest() == expected, path
by_record = {p["record_number"]: p for p in pairs}
def masked(text):
    text = re.sub(r"(?m)^Annotator:[^\n]*$", "Annotator:", text)
    text = re.sub(r"(?m)^Label:[^\n]*$", "Label:", text)
    # Preserve submitted rationales in Git; exclude only quoted notes after a label.
    text = re.sub(r"(?m)(^Label:\n)(?:>[^\n]*\n)+", r"\1", text)
    text = text.replace('Enter your name or agreed identifier after `Annotator:`.', '')
    return re.sub(r"\n{3,}", "\n\n", text).strip()
# Locate exact pristine bytes by the export's recorded template hash.
def pristine(path):
    expected = view_manifest["outputs"][path]
    revisions = git("log", "--format=%H", COMMIT, "--", path).decode().splitlines()
    for revision in revisions:
        body = git("show", f"{revision}:{path}")
        if hashlib.sha256(body).hexdigest() == expected:
            return body.decode("utf-8")
    raise ValueError(f"Pristine template unavailable in local history: {path}")
judgments = {}; summary = []
for path in markdown:
    text = read_text(path)
    assert masked(text) == masked(pristine(path)), f"Content changed: {path}"
    names = re.findall(r"(?m)^Annotator:[ \t]*(.*?)\s*$", text)
    assert len(names) == 1 and names[0].strip(), path
    name = names[0].strip()
    slot = re.search(r"/annotator_([AB])/", path).group(1)
    records = re.findall(r"(?m)^#### Relevance (\d+)\s*\n+Label:[ \t]*([^\n]*)", text)
    passage_ids = [int(x) for x in re.findall(r"(?m)^#### Passage (\d+)$", text)]
    assert [int(n) for n, _ in records] == passage_ids, path
    assert records, path
    for number, label in records:
        number = int(number); label = label.strip()
        assert number in by_record and label in {"0", "1"}, (path, number, label)
        assert (slot, number) not in judgments, (slot, number)
        judgments[slot, number] = {"annotator": name, "label": int(label), "file": path}
    summary.append({"slot": slot, "batch": Path(path).stem, "annotator": name, "filled": len(records)})
for slot in "AB":
    assert {n for s, n in judgments if s == slot} == set(by_record)
for n in by_record:
    assert judgments["A", n]["annotator"].casefold() != judgments["B", n]["annotator"].casefold(), n
for row in summary:
    print(row)
print("PASS: complete labels, distinct identities, unchanged content and frozen input hashes")


## 3. Measure agreement before adjudication

**Expected:** overall agreement ≥80% to meet the stated threshold. Also inspect each actual annotator pair, sample size, label prevalence, and confusion counts. Cohen's κ corrects for chance agreement; the requirement does not specify κ≥0.8. κ is undefined when expected agreement is 1. Do not report one pooled κ across changing annotators.

If agreement is below 80%, preserve and report this round, clarify guidelines, and plan a documented fresh independent round. Adjudicated labels are not independent agreement evidence.


In [ ]:
def agreement(rows):
    n = len(rows)
    counts = Counter((a, b) for a, b in rows)
    observed = sum(a == b for a, b in rows) / n
    pa = sum(a for a, b in rows) / n
    pb = sum(b for a, b in rows) / n
    chance = pa * pb + (1-pa) * (1-pb)
    return {"n": n, "agreement": observed, "kappa": None if chance == 1 else (observed-chance)/(1-chance),
            "positive_rate_left": pa, "positive_rate_right": pb,
            "confusion_00_01_10_11": [counts[x] for x in [(0,0),(0,1),(1,0),(1,1)]]}
by_people = defaultdict(list); all_labels = []
for n in sorted(by_record):
    a, b = judgments["A", n], judgments["B", n]
    all_labels.append((a["label"], b["label"]))
    ordered = sorted([a, b], key=lambda x: x["annotator"].casefold())
    by_people[tuple(x["annotator"] for x in ordered)].append(tuple(x["label"] for x in ordered))
overall = agreement(all_labels)
print("Overall (A/B orientation; pooled kappa deliberately omitted):",
      {k:v for k,v in overall.items() if k != "kappa"})
pair_stats = [{"annotators": list(names), **agreement(rows)} for names, rows in sorted(by_people.items())]
for row in pair_stats: print(row)
AGREEMENT_PASS = overall["agreement"] >= .8
print("80% threshold:", "PASS" if AGREEMENT_PASS else "FAIL — preserve this round and review protocol")


## 4. Prepare human adjudication and audit artifacts

**Expected:** one CSV row per disagreement. A third person outside the original pair fills `adjudicator`, `final_label` (0/1), and `reason`. Do not replace A/B labels. Review agreed records too if a separate documented quality audit is desired; this notebook does not automatically alter them.

This cell creates local outputs and refuses to overwrite an existing adjudication CSV. Preserve the original input commit. Full annotation text is not written into the notebook output; the local CSV includes text for adjudication.


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
ADJ = OUT / "adjudication.csv"
disagreements = [n for n in sorted(by_record) if judgments["A", n]["label"] != judgments["B", n]["label"]]
fields = ["record_number", "question_id", "passage_id", "question", "passage", "annotator_A", "label_A", "annotator_B", "label_B", "adjudicator", "final_label", "reason"]
if not ADJ.exists():
    with ADJ.open("x", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields); writer.writeheader()
        for n in disagreements:
            p = by_record[n]; a,b = judgments["A",n], judgments["B",n]
            writer.writerow({"record_number": n, **{k:p[k] for k in ["question_id","passage_id","question","passage"]},
                             "annotator_A":a["annotator"],"label_A":a["label"],"annotator_B":b["annotator"],"label_B":b["label"],
                             "adjudicator":"", "final_label":"", "reason":""})
else:
    print("Preserving existing adjudication file")
audit = {"schema_version":"v0-proposed", "commit":COMMIT, "pair_count":len(pairs),
         "question_count":200, "overall_agreement":overall["agreement"], "agreement_pass":AGREEMENT_PASS,
         "by_annotator_pair":pair_stats, "disagreements":len(disagreements),
         "input_sha256":{p:hashlib.sha256(read_bytes(p)).hexdigest() for p in markdown + [BASE+"/pairs.jsonl", BASE+"/manifest.json"]}}
(OUT / "agreement.json").write_text(json.dumps(audit, indent=2)+"\n")
print("Disagreements:", len(disagreements), "Adjudication CSV:", ADJ)


## 5. Review scope and provenance before export

**Expected:** explicit human confirmations and a real project role supported by the original freeze record. Do not infer held-out status from directory names. The current pairs preserve official `source_split`; inspect the counts below. Validation-origin data can only be described as held out under the team's documented isolation policy, not as official QASPER test data.

The previous project split mapping was v0-proposed. Consult P3/P6 before assigning the final project role. Set the following values only after review. The export guard is intentional: running all cells does not invent team approval or independent annotation.


In [ ]:
print("Official source split counts:", dict(Counter(p["source_split"] for p in pairs)))
print("Saved retrieval methods:", manifest["settings"]["methods"])
print("Saved top_k:", manifest["settings"]["top_k"])
INDEPENDENCE_CONFIRMED = False
FREEZE_AND_USAGE_REVIEWED = False
PROJECT_ROLE = None  # Set to "development" or "held_out" after actual team review.
REVIEW_NOTE = ""  # Cite the real freeze/usage decision and reviewers; do not fabricate approval.


## 6. Validate adjudication and export provisional gold

**Expected:** this cell stops until all disagreements are adjudicated and the preceding review is complete. It writes per-question labels compatible with the P2 scorer's label shape, plus full original-judgment provenance. `relevance_complete=false` is deliberate: the pool is not exhaustive paper relevance.

The label manifest includes required structural fields, but the scorer also needs the **exact full corpus hash** and compatible retrieval run artifacts. Fill that from the actual corpus used by the saved run; never substitute the pair-file hash. This notebook does not convert saved P3 rankings into the P2 run directory or claim the export alone is scoring-ready.


In [ ]:
assert AGREEMENT_PASS, "Independent agreement is below 80%"
assert INDEPENDENCE_CONFIRMED and FREEZE_AND_USAGE_REVIEWED and REVIEW_NOTE.strip(), "Complete human provenance/scope review"
assert PROJECT_ROLE in {"development", "held_out"}
with ADJ.open(encoding="utf-8", newline="") as f:
    adjudicated = list(csv.DictReader(f))
assert len(adjudicated) == len(disagreements)
assert len({int(r["record_number"]) for r in adjudicated}) == len(adjudicated)
assert {int(r["record_number"]) for r in adjudicated} == set(disagreements)
final = {}; provenance = []
for r in adjudicated:
    n = int(r["record_number"]); p = by_record[n]
    assert all(r[k] == str(p[k]) for k in ["question_id","passage_id","question","passage"]), n
    for slot in "AB":
        assert r["annotator_"+slot] == judgments[slot,n]["annotator"] and r["label_"+slot] == str(judgments[slot,n]["label"]), n
    assert r["final_label"] in {"0","1"} and r["reason"].strip(), n
    assert r["adjudicator"].strip() and r["adjudicator"].strip().casefold() not in {judgments[s,n]["annotator"].casefold() for s in "AB"}, n
    final[n] = int(r["final_label"])
for n,p in sorted(by_record.items()):
    if n not in final: final[n] = judgments["A",n]["label"]
    provenance.append({"record_number":n,"question_id":p["question_id"],"passage_id":p["passage_id"],
                       "A":judgments["A",n],"B":judgments["B",n],"final_label":final[n]})
grouped = defaultdict(list)
for n,p in by_record.items(): grouped[p["question_id"]].append((n,p))
labels = []
for qid, rows in sorted(grouped.items()):
    assert len({p["paper_id"] for n,p in rows}) == 1
    labels.append({"question_id":qid,"paper_id":rows[0][1]["paper_id"],"status":"mapped","relevance_complete":False,
                   "relevant_passage_ids":[p["passage_id"] for n,p in rows if final[n]==1],
                   "judged_passage_ids":[p["passage_id"] for n,p in rows]})
def write_jsonl_new(path, rows):
    with path.open("x", encoding="utf-8") as f:
        for row in rows: f.write(json.dumps(row, ensure_ascii=False)+"\n")
label_path = OUT / "final_labels.jsonl"
write_jsonl_new(label_path, labels)
write_jsonl_new(OUT / "original_and_final_judgments.jsonl", provenance)
label_manifest = {"schema_version":"v0-proposed", "labels_sha256":hashlib.sha256(label_path.read_bytes()).hexdigest(),
    "corpus_sha256":None, "source":{"commit":COMMIT,"review_note":REVIEW_NOTE},
    "mapping":"record_number -> frozen pairs.jsonl -> question_id/passage_id",
    "annotation_coverage":{"unique_pairs":len(pairs),"complete_paper_relevance":False},
    "question_scope":{"question_ids":sorted(grouped),"project_role":PROJECT_ROLE}}
with (OUT / "final_labels_manifest.DRAFT.json").open("x") as f: json.dump(label_manifest,f,indent=2)
print("Exported",len(labels),"question label rows. Manifest is DRAFT until exact corpus hash is supplied and validated.")


## Completion checklist

- [ ] All 1,853 pairs have two independent binary judgments from distinct people.
- [ ] Frozen source hashes and pristine-template content checks passed.
- [ ] Original labels retained; agreement reported before adjudication, overall and by actual pair.
- [ ] ≥80% independent agreement established without deleting difficult examples.
- [ ] Every disagreement adjudicated by a third person with a reason.
- [ ] Actual data-use history and freeze record reviewed; project role and v0-proposed status described accurately.
- [ ] Exact full corpus hash supplied and verified before scoring.
- [ ] Saved rankings retain original ranks, method names and IDs; scoring adapter validated separately.
- [ ] Report pool recall/nDCG and judged coverage; do not treat unjudged passages as negatives.
- [ ] Score K=1,3,5 for this top-5 pool. K=10 requires additional saved retrieval depth and coverage review.
- [ ] Notebook outputs cleared before sharing; generated artifacts remain local.


## Observed snapshot and retrieval handoff (4 November 2026)

Checked commit `1c6953f41d775bcc76fad73a8c5a1617ced261b9`: all 3,706 labels are filled; 1,463/1,853 agree (78.953%), with 390 disagreements. This is below the required 80%. The original pair Jingshuai Qian / Junlin Yin agrees on 280/551 (50.817%); review the relevance-rule interpretation without assuming either annotator is correct. Do not discard this subset or revise submitted judgments to manufacture agreement.

Official source splits: 1,150 test-origin pairs and 703 validation-origin pairs. Review use history before calling the union held out; report official-split subsets separately as appropriate.

The pool already saves 800 rankings (200 questions × four methods), each with five hits. Score these after gold finalization: no retrieval model rerun is needed for historical quality scores. Score K=1,3,5 from the same outputs. All four methods' top-five outputs contributed to the pool; validate IDs and coverage before scoring. Pool-relative recall is not exhaustive paper recall.

Frozen setup: BM25 baseline; `BAAI/bge-small-en-v1.5` dense baseline; hybrid RRF (`candidate_k=20`, `rrf_k=60`); hybrid plus `BAAI/bge-reranker-base` scoring the complete candidate union. Preserve manifest model revisions, corpus, 365-word chunks / 40-word overlap, and same-paper retrieval scope when reproducing the run.

The manifest records 2,128.656 seconds (35.48 minutes) for one Windows CPU run **including indexing and all four methods**. This is historical setup-wide elapsed time, not a per-method benchmark or an estimate for your Mac. Model-free label/ranking scoring can run locally on CPU. Model execution is also possible locally; benchmark representative development questions after warm-up, then estimate each method as `200 × measured seconds/question` plus model loading/index construction. Measure actual full-run latency separately and state hardware, threads, caching, failures, and warm-up policy.

One frozen configuration yields four method evaluations / 800 question-method results. K values are scoring cutoffs, not extra model runs. A new 2×3 candidate/RRF grid would produce six configurations and potentially 24 method evaluations in the grid runner; tune only on development data, not this labeled evaluation pool. No LLM generator is needed for retrieval evaluation. QA/RAGAS runs are a separate stage.
